# SLM Zero Shot Model Instantiation 

This notebook is intended to test SLM model instantiation, prompting, and finetuning. Essentially "scrap notebook".


Notes:
- Investigate zero-shot vs few-shot learning
- Small ("Compact") Language model: 
    - SmolLM2


Libraries to install (?):
- `transformers`


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

checkpoint = "HuggingFaceTB/SmolLM2-135M-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForCausalLM.from_pretrained(checkpoint).to(device)

messages = [
    {"role": "user", "content": "What is gravity?"}
]

input_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

enc = tokenizer(input_text, return_tensors="pt").to(device)

outputs = model.generate(
    **enc,
    max_new_tokens=100,
    temperature=0.2,
    top_p=0.9,
    do_sample=True
)

new_tokens = outputs[0][enc["input_ids"].shape[1]:]
response = tokenizer.decode(new_tokens, skip_special_tokens=True)

# print("Prompt sent to model:\n", input_text)
# print("\nAssistant response:\n", response)
print(response)

Gravity is a fundamental force of nature that attracts objects with mass towards each other. It is a result of the interaction between mass, energy, and space. According to Newton's law of universal gravitation, every point mass attracts every other point mass by a force that is proportional to the product of their masses and inversely proportional to the square of the distance between them.

In simpler terms, gravity is the force that keeps objects from flying apart, and it is a result of the curvature of spacetime caused by the presence of mass and energy. This curvature is what we experience as gravity, and it is a fundamental aspect of the universe.

Think of it this way: if you were to drop an object from a height, it would fall towards the ground because the object's mass is causing it to curve the space around it. This is why objects fall towards the ground, and why we experience gravity.

Gravity is a powerful force that governs the behavior of objects in the universe, and it p

**To-Do list:**
- Understand Recipe-MPR better for fine tuning purposes
- Decide on fine-tuning strategy for SmolLM2; Prompt formatting and expected output will be important
        - LoRA is going to be important for training and WRITING.
- Set up wrapper for Baseline prompting of SLM


For future reference, we will probably eventually do something like:

```py
messages = [
    {
        "role": "user",
        "content": """User request: I want a quick vegetarian breakfast high in protein.

Options:
A. Bacon and egg sandwich
B. Spinach tofu scramble
C. Chicken breakfast burrito
D. Ham omelet
E. Sausage biscuits

Reply with only one letter: A, B, C, D, or E."""
    }
]
```



### Likely Strategy for fine-tuning:

```
<chat template>
user: I want a vegetarian breakfast high in protein

Options:
A bacon sandwich
B tofu scramble
C chicken burrito
D ham omelet
E sausage biscuits

Reply with only one letter.

assistant: B 
```

During training the model learns:

```given prompt → produce "B"```.

In [13]:
import sys
from pathlib import Path

# notebook is in finetuning/, so repo root is one level up
repo_root = Path.cwd().parent
src_path = repo_root / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print("Repo root:", repo_root)
print("Src path added:", src_path)

from recipe_mpr_qa.formats import build_multiple_choice_prompt
from recipe_mpr_qa.data.loaders import load_dataset

from recipe_mpr_qa.data.loaders import load_dataset, load_split_manifest, get_split_examples

dataset = load_dataset("../data/processed/recipe_mpr_qa.jsonl")
manifest = load_split_manifest("../data/processed/primary_split.json")
train_examples = get_split_examples(dataset, manifest, "train")

def example_to_prompt_completion(example):
    prompt, letter_to_id = build_multiple_choice_prompt(
        query=example.query,
        options=example.options,
    )

    gold_letter = next(
        letter for letter, option_id in letter_to_id.items()
        if option_id == example.answer_option_id
    )

    return {
        "prompt": prompt,
        "completion": gold_letter,
        "example_id": example.example_id,
        "answer_option_id": example.answer_option_id,
        "query_type_flags": dict(example.query_type_flags),
    }

print(example_to_prompt_completion(train_examples[0]))

Repo root: c:\Users\Jagri\Documents\Github\ece1508
Src path added: c:\Users\Jagri\Documents\Github\ece1508\src
{'prompt': 'Given the following recipe preference query, select the best matching recipe.\n\nQuery: I want to make a warm dish containing oysters\n\nOptions:\nA) Simple creamy oyster soup\nB) Seasoned salted crackers shaped like oysters\nC) Creamy clam chowder made with whole milk and baby clams\nD) Tomato mussel soup containing dry white wine\nE) Warm vegetable soup containing tomatoes, peas, corn, carrots, and potatoes\n\nRespond with only the letter (A-E) of the best option.', 'completion': 'A', 'example_id': 'rmpr-0001', 'answer_option_id': '08cb462fdf', 'query_type_flags': {'Specific': False, 'Commonsense': True, 'Negated': False, 'Analogical': False, 'Temporal': False}}
